# Hotel Booking Forecasting

## Project Overview

Forecasts **daily hotel booking count** over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, seasonal tourism peaks, and event-driven spikes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily booking counts, predict the next 14 days. Hotels need booking forecasts for revenue management, staffing, inventory allocation, and dynamic pricing.

## Why This Project Matters

Hotel revenue management is a $300B+ industry where pricing depends on demand forecasts. Overbooking causes guest dissatisfaction; underbooking leaves revenue on the table. Even 1% forecast improvement translates to significant revenue gains.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily hotel bookings
- Weekly pattern (weekend leisure peaks)
- Strong summer and holiday season peaks
- Business travel weekday baseline
- Conference/event spikes

| Column | Description |
|--------|-------------|
| `date` | Date |
| `bookings` | Daily hotel booking count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'bookings'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 150 + 0.05 * t  # gradual growth
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 3: weekly[i] = 10  # Mon-Thu business
    elif dow == 4: weekly[i] = 30  # Friday
    elif dow == 5: weekly[i] = 40  # Saturday peak
    else: weekly[i] = 20  # Sunday

# Summer and holiday peaks
seasonal = 30 * np.sin(2 * np.pi * (t - 90) / 365)
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and 20 <= d <= 31): holiday[i] = 40
    elif m == 11 and 22 <= d <= 28: holiday[i] = 25
    elif m == 3 and 10 <= d <= 20: holiday[i] = 20  # spring break

# Conference/event spikes (periodic)
events = np.zeros(N_POINTS)
for s in range(45, N_POINTS, 90):
    for j in range(min(3, N_POINTS - s)):
        events[s + j] = 30 + np.random.uniform(0, 20)

noise = np.random.normal(0, 8, N_POINTS)
bookings = base + weekly + seasonal + holiday + events + noise
bookings = np.maximum(bookings, 50).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'bookings': bookings})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,bookings
0,2022-01-01,172
1,2022-01-02,138
2,2022-01-03,131
3,2022-01-04,119
4,2022-01-05,126


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('bookings Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("bookings Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

bookings Statistics:
count    730.000000
mean     190.390411
std       27.851796
min      117.000000
25%      170.000000
50%      192.000000
75%      211.000000
max      267.000000
Name: bookings, dtype: float64

CV: 0.146
Skewness: -0.103


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       42.8 | RMSE:       46.0 | MAPE: 20.01%
Seasonal Naive (7)             | MAE:       23.4 | RMSE:       29.2 | MAPE: 11.09%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                             
KernelRidge                        2491.471376 -190.574721  191.481215    0.041435
GaussianProcessRegressor            615.989311  -46.306870   95.152276    0.061126
MLPRegressor                        277.213892  -20.247222   63.768787    0.476937
QuantileRegressor                    39.639582   -1.972276   23.850726    0.065109
DummyRegressor                       35.847817   -1.680601   22.650260    0.011781
PassiveAggressiveRegressor           22.184772   -0.629598   17.660268    0.014886
ExtraTreeRegressor                   20.208200   -0.477554   16.816234    0.013246
OrthogonalMatchingPursuit            16.217465   -0.170574   14.967751    0.010903
KNeighborsRegressor                  14.210623   -0.016202   13.945916    0.012356
DecisionTreeRegressor                14.153178   -0.011783   13.91

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        8.1 | RMSE:        9.5 | MAPE:  4.72%
FLAML Test (catboost)          | MAE:       17.0 | RMSE:       19.8 | MAPE:  7.93%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       30.7 | RMSE:       33.6 | MAPE: 14.41%
SF AutoETS                     | MAE:       31.4 | RMSE:       34.2 | MAPE: 14.79%
SF SeasonalNaive               | MAE:       33.9 | RMSE:       37.1 | MAPE: 16.03%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  8.112277  9.521836  4.720875
FLAML Test (catboost) 16.998986 19.776091  7.933119
   Seasonal Naive (7) 23.428571 29.206164 11.090187
         SF AutoARIMA 30.706585 33.612428 14.409703
           SF AutoETS 31.446608 34.238589 14.789615
     SF SeasonalNaive 33.928570 37.068469 16.027356
   Naive (Last Value) 42.785714 45.994565 20.006560

Top 3:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  8.112277  9.521836  4.720875
FLAML Test (catboost) 16.998986 19.776091  7.933119
   Seasonal Naive (7) 23.428571 29.206164 11.090187


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 16.51, Std: 10.89


## Interpretation and Business Insight

- Hotel bookings peak on weekends (leisure travelers)
- Summer and holiday seasons create major demand surges
- Conference events create periodic multi-day spikes
- Business travel provides a stable weekday baseline
- Cancellation patterns (not modeled) add additional uncertainty

## Limitations

1. Synthetic — real bookings depend on pricing, reviews, competition
2. No pricing/rate features
3. No cancellation modeling
4. No competitor data (OTA pricing)
5. Single property — chains need portfolio optimization

## How to Improve This Project

1. Add dynamic pricing features (own rate vs competitors)
2. Model cancellations and no-shows separately
3. Include event calendar data
4. Add weather forecasts for leisure destinations
5. Build lead-time demand curves for revenue management

## Production Considerations

- Daily booking forecast by room type and rate tier
- Integration with property management systems
- Dynamic pricing engine feeding from forecasts
- Overbooking optimization with no-show prediction

## Common Mistakes

1. Not accounting for cancellations in net booking forecasts
2. Ignoring competitor pricing effects
3. Using aggregate forecasts for room-type decisions
4. Not modeling lead-time booking patterns

## Mini Challenge / Exercises

1. Add a synthetic cancellation rate and model net bookings
2. Model weekend leisure vs weekday business separately
3. Detect conference/event periods in the data
4. Build an occupancy forecast from bookings + cancellations

## Final Summary / Key Takeaways

1. Hotel booking forecasting is fundamental to revenue management
2. Weekend leisure and seasonal tourism drive demand peaks
3. Events create predictable multi-day spikes
4. Cancellation modeling is essential for net demand
5. Dynamic pricing depends on accurate forward-looking demand forecasts

In [18]:
import json
metrics = {
    'project': 'Hotel Booking Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Hotel Booking Forecasting — Complete ===")

Metrics saved.

=== Hotel Booking Forecasting — Complete ===
